# 00 · 스모크 테스트 (preflight)

학습 돌리기 전에 이것만 확인:
1. **경로** — 레포/venv/출력 위치가 맞는가
2. **CUDA + mamba_ssm** — Mamba 커널은 CUDA 전용. 없으면 `acm`/`ours` 전부 죽음
3. **carry parity** — `tests/test_acm_sscp_literal.py` (⚠️ 깨지면 `ours` 학습이 무의미)
4. **태그/커맨드** — 10개 태그가 등록돼 있고 커맨드가 제대로 만들어지는가

이 노트북은 **학습을 돌리지 않는다**(dry-run).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

print('레포      :', cf.v23.PROJECT_ROOT)
print('  src/lerobot :', (cf.v23.SRC_DIR / 'lerobot').is_dir())
print('venv python:', cf.v23.PYTHON, '  존재:', Path(cf.v23.PYTHON).exists())
print('출력      :', cf.OUTPUT_BASE)
print()
print('학습 : %s step | seed %s | lr sweep 없음' % (f'{cf.STEPS:,}', cf.MAIN_SEEDS))
print('eval : %s ckpt x %d rep x %d ep' % (f'{cf.CKPT_STEP:,}', cf.EVAL_REPEATS, cf.EVAL_N_EP))

## 1) CUDA + mamba_ssm (학습 subprocess 가 쓸 venv 기준)

In [ ]:
import subprocess
probe = (
    "import torch;"
    "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),"
    "'|', torch.cuda.device_count(), 'GPU');"
    "import mamba_ssm; print('mamba_ssm OK')"
)
r = subprocess.run([cf.v23.PYTHON, '-c', probe], capture_output=True, text=True)
print(r.stdout or r.stderr)
print('사용 가능 GPU:', cf.available_gpus(list(range(8))))

## 1b) 렌더 백엔드 — **EGL(GPU) 이어야 함**
학습·eval 커맨드 앞에 붙는 prefix 를 그대로 출력한다. `MUJOCO_GL=egl` 이 보여야 정상.
`osmesa` 가 보이면 CPU 소프트웨어 렌더 → eval 이 수 배 느려짐 (`RENDER_BACKEND` 를 export 하지 말 것).

In [ ]:
import os
pre = cf.v23._render_prefix()
print('render prefix:')
for kv in pre.split():
    print('   ', kv)
gl = 'EGL' if 'MUJOCO_GL=egl' in pre else ('OSMESA' if 'osmesa' in pre else '??')
print()
print('=> 렌더 백엔드:', gl, '(EGL 이어야 정상 — OSMESA 는 CPU 렌더라 수 배 느림)')
print('   RENDER_BACKEND env:', os.environ.get('RENDER_BACKEND'))
# 각 run 은 prefix 뒤에 MUJOCO_EGL_DEVICE_ID={gpu} 를 다시 써서 자기 GPU 로 렌더(prefix 의 0 을 덮어씀)
print('   GPU별 렌더:', [f'gpu{g} -> MUJOCO_EGL_DEVICE_ID={g}' for g in range(3)])


## 2) carry parity 테스트 (★가장 중요)
`ours` 의 핵심 = Mamba-1 selective scan 에 **초기 상태를 closed-form 으로 주입**하는 carry.
이게 틀리면 학습 24잡이 통째로 헛돈다.

In [ ]:
test = cf.v23.PROJECT_ROOT / 'tests' / 'test_acm_sscp_literal.py'
r = subprocess.run([cf.v23.PYTHON, str(test)], capture_output=True, text=True,
                   cwd=str(cf.v23.PROJECT_ROOT))
print(r.stdout[-2000:] or r.stderr[-2000:])
print('\n=> ', 'PASS' if r.returncode == 0 else 'FAIL (학습 시작하지 말 것)')

## 3) 태그 + 학습/eval 커맨드 (dry-run)

In [ ]:
cf.check_tags()
print()
TASK = cf.MAIN_SIM
for t in cf.TRAIN_TAGS:
    c = cf.make_train_cmd(t, seed=0, task=TASK, gpu_id=0)
    flags = [p for p in c.split() if p.startswith(('--policy.optimizer_lr', '--steps', '--policy.chunk_size',
                                                   '--policy.horizon', '--use_chunk_pairs'))]
    print(f'{t:<10} ' + '  '.join(flags))
print()
print('[ours 전체]')
print(cf.make_train_cmd('ours', seed=0, task=TASK, gpu_id=0))

## 4) 다음 단계
- 위 3개 전부 OK → `01_train_ours` (150k · seed 4개)
- 그 다음 `02_eval_ours` (150k ckpt × 5 rep) → `10_report_jerk` / `05_train_ablation` / `11_efficiency`
